# 05b — GraphRAG knowledge graph over the company universe

Builds a property graph over every organisation in `conf/companies.yaml`, stores it as two
Delta tables, and exposes it to the Multi-Agent Supervisor as **SQL Unity Catalog functions**.

---

## Architecture

```
┌─────────────────────────────────┐
│  Supervisor Agent (notebook 04) │
│  routes queries to child agents │
└───────────┬─────────────────────┘
            │  UC function tools
            ▼
┌───────────────────────────────────────┐
│  get_company_summary(company)         │
│  compare_companies(companies)         │
└───────────┬───────────────────────────┘
            │  Spark SQL
            ▼
┌───────────────────────────────────────┐
│  Unity Catalog Delta Tables           │
│    • graphrag_vertices                │
│    • graphrag_edges                   │
└───────────────────────────────────────┘
```

### Why UC functions and not an MCP server

An earlier version of this notebook deployed a FastMCP server as a Databricks App. That
approach is unusable through the Agent Bricks MCP proxy on streaming requests
(`httpx.ResponseNotRead`; see ML-63338), while UC function tools register reliably. The app
and connection cells were removed. Don't reintroduce an app-hosted MCP server for this.

### Why the graph is built from the manifest

This notebook used to derive its company list from whatever was in `ticker_data`. That
silently excluded every organisation without a listed share price — which, in this corpus,
is a third of them. American Honda Finance files a full 10-K and has no ticker; the health
systems and private companies have neither ticker nor filing. They are also the subjects of
the most interesting graph questions ("who owns FordDirect?"). So the universe now comes
from `conf/companies.yaml` and market data is joined on where it exists.

---

## Graph Schema — Nodes

### Company
One per organisation in the manifest, plus the seven original Mag7 names carried by
`ticker_data`. Fundamentals are present only for listed companies; the rest carry nulls.

| Property | Type | Example | Description |
|----------|------|---------|-------------|
| `id` | string | `Ford Motor Company` | Full company name — the node identity |
| `ticker` | string | `F` | Symbol, or NULL when the organisation is not listed |
| `market_cap` / `beta` / `pe_trailing` / `pe_forward` / `ev_ebitda` | string | | Latest snapshot, NULL when unlisted |

### TradingDay
One per (symbol × trading date), ~250 per listed company.

| Property | Type | Example |
|----------|------|---------|
| `id` | string | `F_2026-06-30` |
| `ticker` | string | `F` |
| `date` / `price_close` / `volume` / `daily_return` | string | |

### Hub nodes
`Index` (ERA Universe, MAG7) · `Sector` (one per manifest sector) · `BetaTier` ·
`ValuationTier` · `DocumentSource` (real filing vs generated document).

---

## Graph Schema — Edges

| Relationship | Source → Target | Purpose |
|-------------|-----------------|---------|
| **TRADED_ON** | Company → TradingDay | Links a company to each of its trading days |
| **NEXT_DAY** | TradingDay → TradingDay | Temporal chain per symbol, for streak/momentum queries |
| **MEMBER_OF** | Company → Index | Universe and Mag7 membership |
| **BELONGS_TO** | Company → Sector | Sector grouping, from the manifest |
| **SUBSIDIARY_OF** / **JOINT_VENTURE_OF** | Company → Company | Real ownership, stated in the companies' own filings |
| **HAS_DOCUMENT_SOURCE** | Company → DocumentSource | Whether the company's corpus document is a real filing or generated |
| **HAS_BETA** / **HAS_VALUATION** | Company → tier | Risk and valuation buckets, listed companies only |
| **CORRELATED_WITH** | Company → Company | Weighted, daily-return correlation ≥ 0.6 |

**Ownership edges are the only ones asserting a real-world fact.** Sector and tier edges are
derived from data in the manifest and the price table; correlation is computed. Keep it that
way — an invented ownership edge is indistinguishable from a real one once it is in the graph.

**Document provenance is an edge on purpose.** Eight organisations have generated documents
rather than filings, and making that visible in the graph means the distinction survives into
anything that reads it, rather than living only in a filename convention.

---

## UC Functions Exposed

| Function | Signature | Notes |
|----------|-----------|-------|
| `get_company_summary` | `get_company_summary(company: str)` | Accepts a ticker **or** a company name. Most of this universe has no ticker, and an agent asked about "Providence Health" will pass exactly that. Resolved through an alias map built from the manifest. |
| `compare_companies` | `compare_companies(companies: str)` | Comma-separated tickers or names. Unlisted organisations appear with `n/a` fundamentals rather than being dropped. |

---

## Cell Execution Order

| Cell | Action | Idempotent? |
|------|--------|-------------|
| 1 | Install `fastmcp` (legacy; not used by the UC-function path) | Yes |
| 2 | Load config, init SDK client | Yes |
| 3 | Build graph DataFrames from the manifest + `ticker_data` | Yes |
| 4 | Write `graphrag_vertices` / `graphrag_edges` | Yes (overwrite) |
| 5 | Append hubs, ownership, provenance, tiers, correlations | Yes (append after overwrite) |
| 6 | Create the two UC functions | Yes |
| 7 | Smoke tests | Yes |
| 9 | Network visualisation | Yes |

---

## Prerequisites

1. Run **`00_tables_setup`** — creates `ticker_data`
2. Run the corpus scripts if you want documents to match the graph:
   `resources/fetch_filings.py` and `resources/generate_synthetic_docs.py`
3. **`../config.py`** must define `catalog`, `schema`, `app_resource_suffix`
4. `CREATE TABLE` and `CREATE FUNCTION` on the target schema

## Troubleshooting

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| A company is missing from the graph | Not in `conf/companies.yaml` | Add it there; everything else follows |
| Unlisted companies tagged high-beta | `isNotNull` filter removed from cell 5 | Smoke test 5 asserts against exactly this |
| `get_company_summary('Providence')` returns nothing | Alias map stale after a manifest rename | Re-run cell 6 |
| Functions return empty | `graphrag_vertices` missing or empty | Re-run cells 3–5 |

In [0]:
# --- Dependencies ---
# fastmcp is only needed for local testing/validation of the MCP server. The Databricks App installs its
# own copy from requirements.txt (see cell 6). The OntoBricks/ontos package is NOT installed here because
# it runs inside the Databricks App runtime, not in this notebook's compute context.
%pip install fastmcp --quiet
dbutils.library.restartPython()

In [0]:
# --- Imports & Configuration ---
# Standard libraries plus PySpark for graph DataFrame construction (cells 4-5) and the Databricks SDK for
# REST API calls to manage Apps and Connections (cells 7-8).
import sys
import os
import json
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from databricks.sdk import WorkspaceClient

# Load shared project config (catalog, schema, app_resource_suffix, sa_name). These variables are defined
# in ../config.py and shared across all setup notebooks.
sys.path.append('..')
exec(open('../config.py').read())

CATALOG = catalog
SCHEMA = schema

# GraphRAG is SQL UC functions over Delta tables. There is no app or MCP connection.

# WorkspaceClient auto-authenticates using the notebook's execution context. We use w.api_client.do() for
# REST calls throughout because the SDK's high-level methods (w.apps.*, w.connections.*) are incomplete on
# serverless compute.
w = WorkspaceClient()
print(f"Catalog: {CATALOG}")
print(f"Schema:  {SCHEMA}")


In [ ]:
# --- Build Graph DataFrames ---
# Transform the flat ticker_data table (created by 00_tables_setup) into a property-graph representation
# with two vertex types and two edge types, then add every company the manifest knows about.
#
# THE COMPANY UNIVERSE COMES FROM THE MANIFEST, NOT FROM ticker_data
# ------------------------------------------------------------------
# This cell used to derive its company list from whatever happened to be in ticker_data. That silently
# excluded every organisation without a listed share price - American Honda Finance files a full 10-K but
# has no ticker, and the nonprofits and private companies have neither. They belong in the graph: the
# interesting questions ("who owns FordDirect", "which health systems are covered") are exactly the ones
# about entities with no market data. So companies come from conf/companies.yaml and market data is
# joined on where it exists.
#
# NODE IDENTITY: id = company name, ticker = symbol
# -------------------------------------------------
# Previously both were company_name, which was the ticker symbol, so graph nodes were called "AAPL".
# Now ticker_data carries real names, so the two are separated: the node is identified by its name and
# carries its symbol as a property. Everything that references a Company node uses the name.

from era.ingest.companies import load_companies, sectors  # noqa: E402

COMPANIES = load_companies()
print(f"Manifest: {len(COMPANIES)} companies "
      f"({sum(c.is_filer for c in COMPANIES)} SEC filers, "
      f"{sum(c.is_synthetic for c in COMPANIES)} with generated documents)")

# ── Source Data ──
# ticker_data has one row per (company × trading day) with columns: date, symbol, company_name,
# price_open, price_close, volume, pe_trailing, pe_forward, peg, ev_ebitda, market_cap, beta
ticker_df = spark.table(f"{CATALOG}.{SCHEMA}.ticker_data")
print(f"Ticker data: {ticker_df.count()} rows, {ticker_df.select('symbol').distinct().count()} symbols")

# ── Company Vertices: market-backed ──
# One node per priced company, carrying the latest fundamental snapshot from Yahoo Finance.
priced_df = ticker_df.select(
    F.col("company_name").alias("id"),
    F.lit("Company").alias("type"),
    F.col("symbol").alias("ticker"),
    "market_cap", "beta", "pe_trailing", "pe_forward", "ev_ebitda"
).dropDuplicates(["id"])

priced_names = {row.id for row in priced_df.select("id").collect()}

# ── Company Vertices: manifest-only ──
# Organisations with no price feed: unlisted registrants and everything with a generated document. They
# get a node with null fundamentals rather than no node at all, so relationship questions still resolve.
unpriced = [
    (c.name, "Company", c.ticker, None, None, None, None, None)
    for c in COMPANIES if c.name not in priced_names
]
if unpriced:
    unpriced_df = spark.createDataFrame(
        unpriced,
        "id string, type string, ticker string, market_cap double, beta double, "
        "pe_trailing double, pe_forward double, ev_ebitda double",
    )
    companies_df = priced_df.unionByName(unpriced_df)
else:
    companies_df = priced_df

print(f"Company vertices: {companies_df.count()} "
      f"({len(priced_names)} with market data, {len(unpriced)} without)")

# ── TradingDay Vertices ──
# One per (symbol × date). The composite ID "SYMBOL_YYYY-MM-DD" stays keyed on the symbol because it must
# be short and unique; only Company nodes use the full name.
trading_days_df = ticker_df.select(
    F.concat_ws("_", "symbol", F.date_format("date", "yyyy-MM-dd")).alias("id"),
    F.lit("TradingDay").alias("type"),
    F.col("symbol").alias("ticker"),
    F.col("date").alias("date"),
    "price_open", "price_close", "volume",
    F.round((F.col("price_close") - F.col("price_open")) / F.col("price_open") * 100, 4).alias("daily_return")
)

# ── TRADED_ON Edges ──
# Company (by name) → its TradingDay nodes (by symbol_date).
traded_on_edges = ticker_df.select(
    F.col("company_name").alias("src"),
    F.concat_ws("_", "symbol", F.date_format("date", "yyyy-MM-dd")).alias("dst"),
    F.lit("TRADED_ON").alias("relationship")
)

# ── NEXT_DAY Edges ──
# Temporal chain: each TradingDay points to the next trading day for the SAME symbol, so streak and
# momentum questions traverse the graph instead of needing window functions.
day_window = Window.partitionBy("symbol").orderBy("date")
with_next = ticker_df.withColumn("next_date", F.lead("date").over(day_window))
next_day_edges = with_next.filter(F.col("next_date").isNotNull()).select(
    F.concat_ws("_", "symbol", F.date_format("date", "yyyy-MM-dd")).alias("src"),
    F.concat_ws("_", "symbol", F.date_format("next_date", "yyyy-MM-dd")).alias("dst"),
    F.lit("NEXT_DAY").alias("relationship")
)

print(f"\nGraph components:")
print(f"  Company vertices: {companies_df.count()}")
print(f"  TradingDay vertices: {trading_days_df.count()}")
print(f"  TRADED_ON edges: {traded_on_edges.count()}")
print(f"  NEXT_DAY edges: {next_day_edges.count()}")

In [0]:
# --- Persist Graph as Delta Tables ---
# We flatten all vertex types into a single table with a union-compatible schema. Properties that don't
# apply to a given vertex type are stored as NULL. All property columns are cast to STRING for uniform
# storage. A `name` column labels the hub nodes added in the enrichment cell below.
#
# This "wide + sparse" layout is simpler than separate tables per vertex type and works well because the
# pro app issues targeted SQL with WHERE type = '...'.

vertices_table = f"{CATALOG}.{SCHEMA}.graphrag_vertices"
edges_table = f"{CATALOG}.{SCHEMA}.graphrag_edges"

# ── Vertices: union Company + TradingDay nodes ──
# Company nodes carry fundamentals (market_cap, beta, pe_*) but no date/price props. TradingDay nodes carry
# date/price/volume/return but no fundamental props. `name` is NULL for both (used only by hub nodes).
all_vertices = companies_df.select("id", "type", "ticker",
    F.col("market_cap").cast("string").alias("prop_market_cap"),
    F.col("beta").cast("string").alias("prop_beta"),
    F.col("pe_trailing").cast("string").alias("prop_pe_trailing"),
    F.col("pe_forward").cast("string").alias("prop_pe_forward"),
    F.col("ev_ebitda").cast("string").alias("prop_ev_ebitda"),
    F.lit(None).cast("string").alias("prop_date"),
    F.lit(None).cast("string").alias("prop_price_close"),
    F.lit(None).cast("string").alias("prop_volume"),
    F.lit(None).cast("string").alias("prop_daily_return"),
    F.lit(None).cast("string").alias("name")
).unionByName(
    trading_days_df.select("id", "type", "ticker",
        F.lit(None).cast("string").alias("prop_market_cap"),
        F.lit(None).cast("string").alias("prop_beta"),
        F.lit(None).cast("string").alias("prop_pe_trailing"),
        F.lit(None).cast("string").alias("prop_pe_forward"),
        F.lit(None).cast("string").alias("prop_ev_ebitda"),
        F.col("date").cast("string").alias("prop_date"),
        F.col("price_close").cast("string").alias("prop_price_close"),
        F.col("volume").cast("string").alias("prop_volume"),
        F.col("daily_return").cast("string").alias("prop_daily_return"),
        F.lit(None).cast("string").alias("name")
    )
)

# ── Edges: union TRADED_ON + NEXT_DAY relationships ──
# Both edge types share the same (src, dst, relationship) schema. `weight` is NULL here; the enrichment
# cell uses it to carry correlation strength.
all_edges = traded_on_edges.unionByName(next_day_edges).withColumn(
    "weight", F.lit(None).cast("double")
)

# Write with overwrite mode so the notebook is fully idempotent. Delta gives us ACID guarantees + time-travel.
all_vertices.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(vertices_table)
all_edges.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(edges_table)

print(f"✅ Wrote {vertices_table} ({all_vertices.count()} rows)")
print(f"✅ Wrote {edges_table} ({all_edges.count()} rows)")

In [ ]:
# --- Enrich the graph: sectors, ownership, document provenance, risk/valuation tiers, correlation web ---
# Beyond the Company→TradingDay star, we add company-level structure so the graph has several "centers of
# gravity" and cross-company relationships the pro app visualises:
#   • Index         : IDX::ERA_UNIVERSE (everything in the manifest) and IDX::MAG7 (the original seven)
#   • Sector        : sector hubs companies BELONG_TO, taken from the manifest
#   • Ownership     : SUBSIDIARY_OF / JOINT_VENTURE_OF, the only edges here asserting a real-world fact
#   • DocumentSource: whether a company's corpus document is a real filing or a generated one
#   • BetaTier / ValuationTier : risk + valuation buckets, only for companies that have fundamentals
#   • CORRELATED_WITH : weighted edges between companies whose daily returns correlate >= 0.6
# These rows are *appended* to the tables written above (which overwrite), so re-running cells 4+5 stays
# idempotent.
#
# WHY DOCUMENT PROVENANCE IS IN THE GRAPH
# ---------------------------------------
# Eight companies have generated documents rather than filings. Making that a visible edge means the
# distinction survives into anything reading the graph, instead of living only in a filename convention.

PROP_COLS = ["prop_market_cap", "prop_beta", "prop_pe_trailing", "prop_pe_forward",
             "prop_ev_ebitda", "prop_date", "prop_price_close", "prop_volume", "prop_daily_return"]

MAG7_SYMBOLS = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "TSLA"]
MAG7_SECTORS = {
    "AAPL": "Technology", "MSFT": "Technology", "NVDA": "Technology",
    "GOOGL": "Communication Services", "META": "Communication Services",
    "AMZN": "Consumer Cyclical", "TSLA": "Consumer Cyclical",
}

# Resolve the names the graph actually uses for the legacy seven, whose rows come from Yahoo Finance
# rather than the manifest.
symbol_to_name = {
    r.symbol: r.company_name
    for r in ticker_df.select("symbol", "company_name").distinct().collect()
}
mag7_names = [symbol_to_name[s] for s in MAG7_SYMBOLS if s in symbol_to_name]

# name → sector, covering both sources of companies
sector_by_name = {c.name: c.sector for c in COMPANIES}
for symbol, sector in MAG7_SECTORS.items():
    if symbol in symbol_to_name:
        sector_by_name[symbol_to_name[symbol]] = sector

all_sectors = sorted({s for s in sector_by_name.values() if s})

# ── Hub vertices (indices / sectors / tiers / document sources) ──
hub_specs = [
    ("IDX::ERA_UNIVERSE", "Index", "ERA Universe"),
    ("IDX::MAG7", "Index", "MAG7"),
    ("SRC::REAL_FILING", "DocumentSource", "Real filing"),
    ("SRC::GENERATED", "DocumentSource", "Generated document"),
    ("BETA::Low", "BetaTier", "Low Beta"),
    ("BETA::Moderate", "BetaTier", "Moderate Beta"),
    ("BETA::High", "BetaTier", "High Beta"),
    ("VAL::Value", "ValuationTier", "Value"),
    ("VAL::Core", "ValuationTier", "Core"),
    ("VAL::Premium", "ValuationTier", "Premium"),
] + [(f"SEC::{s}", "Sector", s) for s in all_sectors]

hub_vertices = spark.createDataFrame(hub_specs, ["id", "type", "name"]).withColumn(
    "ticker", F.lit(None).cast("string")
)
for c in PROP_COLS:
    hub_vertices = hub_vertices.withColumn(c, F.lit(None).cast("string"))
hub_vertices = hub_vertices.select("id", "type", "ticker", *PROP_COLS, "name")
hub_vertices.write.format("delta").mode("append").saveAsTable(vertices_table)


def _edges(pairs, relationship):
    """(src, dst) tuples -> the edge schema, or None when there is nothing to add."""
    if not pairs:
        return None
    return (spark.createDataFrame(pairs, "src string, dst string")
            .withColumn("relationship", F.lit(relationship))
            .withColumn("weight", F.lit(None).cast("double")))


# ── Sector membership ──
belongs_to = _edges(
    [(name, f"SEC::{sector}") for name, sector in sector_by_name.items() if sector],
    "BELONGS_TO",
)

# ── Index membership ──
member_of = _edges(
    [(c.name, "IDX::ERA_UNIVERSE") for c in COMPANIES]
    + [(n, "IDX::MAG7") for n in mag7_names],
    "MEMBER_OF",
)

# ── Ownership: the only edges here that assert something about the real world ──
# Both are stated in the companies' own filings: American Honda Finance is an indirect wholly-owned
# subsidiary of Honda Motor, and FordDirect is a joint venture of Ford Motor Company and its dealers.
by_slug = {c.slug: c for c in COMPANIES}
ownership = _edges(
    [(c.name, by_slug[rel.target].name)
     for c in COMPANIES for rel in c.relationships if rel.target in by_slug],
    "OWNED_BY",
)
# Keep the specific relationship type rather than flattening everything to OWNED_BY.
if ownership is not None:
    rel_type = {(c.name, by_slug[rel.target].name): rel.type.upper()
                for c in COMPANIES for rel in c.relationships if rel.target in by_slug}
    mapping = F.create_map([F.lit(x) for kv in rel_type.items()
                            for x in (f"{kv[0][0]}->{kv[0][1]}", kv[1])])
    ownership = ownership.withColumn(
        "relationship", mapping[F.concat_ws("->", F.col("src"), F.col("dst"))]
    )

# ── Document provenance ──
# The legacy seven are backed by the real 10-K/10-Q PDFs committed under data/, so they count as filed.
has_source = _edges(
    [(c.name, "SRC::GENERATED" if c.is_synthetic else "SRC::REAL_FILING") for c in COMPANIES]
    + [(n, "SRC::REAL_FILING") for n in mag7_names],
    "HAS_DOCUMENT_SOURCE",
)

# ── Risk (beta) + valuation (forward P/E) tiers ──
# Restricted to companies that actually have fundamentals. Without the isNotNull filter every unpriced
# company falls through to the otherwise() branch and gets labelled high-beta and premium-valued on the
# strength of having no data at all.
has_beta = companies_df.where(F.col("beta").isNotNull()).select(
    F.col("id").alias("src"),
    F.when(F.col("beta") < 1.15, "BETA::Low")
     .when(F.col("beta") <= 1.5, "BETA::Moderate")
     .otherwise("BETA::High").alias("dst"),
).withColumn("relationship", F.lit("HAS_BETA")).withColumn("weight", F.lit(None).cast("double"))

has_valuation = companies_df.where(F.col("pe_forward").isNotNull()).select(
    F.col("id").alias("src"),
    F.when(F.col("pe_forward") < 30, "VAL::Value")
     .when(F.col("pe_forward") <= 40, "VAL::Core")
     .otherwise("VAL::Premium").alias("dst"),
).withColumn("relationship", F.lit("HAS_VALUATION")).withColumn("weight", F.lit(None).cast("double"))

# ── Correlation web: pairwise daily-return correlation, kept when >= 0.6, weighted ──
# Mapped back to company names so the edges connect Company nodes rather than dangling on symbols.
name_map = spark.createDataFrame(
    [(s, n) for s, n in symbol_to_name.items()], "symbol string, company_name string"
)
returns = (trading_days_df.select(F.col("ticker").alias("symbol"), "date", "daily_return")
           .join(name_map, "symbol").select("company_name", "date", "daily_return"))
a = returns.alias("a")
b = returns.alias("b")
pairs = a.join(b, (F.col("a.date") == F.col("b.date")) &
                  (F.col("a.company_name") < F.col("b.company_name")))
correlated = pairs.groupBy(F.col("a.company_name").alias("src"),
                           F.col("b.company_name").alias("dst")) \
    .agg(F.round(F.corr("a.daily_return", "b.daily_return"), 3).alias("weight")) \
    .where(F.col("weight") >= 0.6) \
    .withColumn("relationship", F.lit("CORRELATED_WITH")) \
    .select("src", "dst", "relationship", "weight")

parts = [df for df in (member_of, belongs_to, ownership, has_source,
                       has_beta, has_valuation, correlated) if df is not None]
enrichment_edges = parts[0]
for df in parts[1:]:
    enrichment_edges = enrichment_edges.unionByName(df)
enrichment_edges.write.format("delta").mode("append").saveAsTable(edges_table)

print(f"✅ Appended {hub_vertices.count()} hub vertices "
      f"({len(all_sectors)} sectors, 2 indices, 2 document sources, 6 tiers)")
print(f"✅ Appended {enrichment_edges.count()} structural + correlation edges")
print(f"   Ownership edges: {ownership.count() if ownership is not None else 0}")
print(f"   Correlation edges: {correlated.count()} (return corr >= 0.6)")

In [ ]:
# --- Create GraphRAG SQL UC functions (replaces the FastMCP server approach) ---
# The Supervisor consumes the knowledge graph through SQL Unity Catalog functions over
# graphrag_vertices, NOT an external MCP server. External MCP on Databricks Apps is
# unusable through the Agent Bricks MCP proxy on streaming requests (httpx
# ResponseNotRead; see ML-63338). UC function tools register reliably. Notebook 04 adds
# these as Supervisor tools; notebook 06 grants the app service principal EXECUTE.
#
# LOOKUP TAKES A TICKER *OR* A NAME
# ---------------------------------
# Most of this universe has no ticker to look up by - the health systems, the private
# companies, American Honda Finance. And an agent asked about "Providence Health" will
# pass exactly that, not a symbol. So both functions resolve their argument through an
# alias map built from the manifest before matching, and match exactly afterwards.
# Exact matching matters: a LIKE would let "Ford" match both Ford Motor Company and
# FordDirect, and the aggregate below would silently merge the two into one answer.
VERTICES = f"`{CATALOG}`.`{SCHEMA}`.graphrag_vertices"


def _sql_str(value: str) -> str:
    return "'" + str(value).replace("'", "''") + "'"


# alias (lowercased) → canonical company name, from every name, alias and ticker we know
alias_pairs: dict[str, str] = {}
for c in COMPANIES:
    for label in (*c.all_names, c.slug.replace("_", " "), c.ticker):
        if label:
            alias_pairs[str(label).lower()] = c.name
for symbol, name in symbol_to_name.items():
    alias_pairs.setdefault(symbol.lower(), name)
    alias_pairs.setdefault(name.lower(), name)

ALIAS_MAP = "MAP(" + ", ".join(
    f"{_sql_str(k)}, {_sql_str(v)}" for k, v in sorted(alias_pairs.items())
) + ")"

# Names the tool description advertises, so the Supervisor knows the universe is broader
# than seven tech stocks and includes organisations with no ticker at all.
EXAMPLES = ", ".join([c.name for c in COMPANIES[:6]])

spark.sql(f"""
CREATE OR REPLACE FUNCTION `{CATALOG}`.`{SCHEMA}`.get_company_summary(p_company STRING)
RETURNS STRING
COMMENT 'Knowledge-graph summary for one organisation: fundamentals (market cap, beta, trailing/forward P/E, EV/EBITDA) where the company is publicly listed, plus its 10 most recent trading days (date | close | volume). Accepts either a ticker (NVDA, F, PFE, UAL) or a company name ({EXAMPLES}). Many covered organisations are unlisted - nonprofit health systems, private companies and debt-only registrants - and correctly return no market data.'
RETURN (
  WITH resolved AS (SELECT COALESCE(element_at({ALIAS_MAP}, LOWER(TRIM(p_company))), TRIM(p_company)) AS name)
  SELECT CONCAT_WS(char(10),
    CONCAT('=== ', (SELECT name FROM resolved), ' (knowledge graph) ==='),
    CASE WHEN MAX(CASE WHEN type='Company' THEN prop_market_cap END) IS NULL
         THEN 'fundamentals: none - this organisation is not publicly listed'
         ELSE CONCAT('market_cap=', MAX(CASE WHEN type='Company' THEN prop_market_cap END),
           ' beta=', MAX(CASE WHEN type='Company' THEN prop_beta END),
           ' pe_trailing=', MAX(CASE WHEN type='Company' THEN prop_pe_trailing END),
           ' pe_forward=', MAX(CASE WHEN type='Company' THEN prop_pe_forward END),
           ' ev_ebitda=', MAX(CASE WHEN type='Company' THEN prop_ev_ebitda END)) END,
    'recent_days (date | close | volume):',
    ARRAY_JOIN(SLICE(ARRAY_SORT(
        COLLECT_LIST(CASE WHEN type='TradingDay' THEN CONCAT_WS(' | ', prop_date, prop_price_close, prop_volume) END),
        (l, r) -> CASE WHEN l > r THEN -1 WHEN l < r THEN 1 ELSE 0 END), 1, 10), char(10))
  )
  FROM {VERTICES}
  WHERE id = (SELECT name FROM resolved)
     OR (ticker IS NOT NULL AND ticker = (
           SELECT MAX(ticker) FROM {VERTICES}
           WHERE type='Company' AND id = (SELECT name FROM resolved)))
)
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION `{CATALOG}`.`{SCHEMA}`.compare_companies(p_companies STRING)
RETURNS STRING
COMMENT 'Compare organisations from the knowledge graph by fundamentals (market cap, beta, trailing/forward P/E, EV/EBITDA), sorted by market cap. Input: a comma-separated list of tickers or company names, e.g. "Ford Motor Company,Toyota Motor Corporation" or "PFE,MRK,AMGN". Unlisted organisations appear with no market data rather than being omitted.'
RETURN (
  WITH wanted AS (
    SELECT COALESCE(element_at({ALIAS_MAP}, LOWER(TRIM(x))), TRIM(x)) AS name
    FROM (SELECT EXPLODE(SPLIT(p_companies, ',')) AS x)
  )
  SELECT CONCAT_WS(char(10),
    '=== Company comparison (knowledge graph) ===',
    'company | ticker | market_cap | beta | pe_trailing | pe_forward | ev_ebitda',
    ARRAY_JOIN(TRANSFORM(ARRAY_SORT(
        COLLECT_LIST(NAMED_STRUCT(
          'mc', COALESCE(CAST(prop_market_cap AS DOUBLE), -1),
          'line', CONCAT_WS(' | ', id, COALESCE(ticker, 'unlisted'),
                            COALESCE(prop_market_cap, 'n/a'), COALESCE(prop_beta, 'n/a'),
                            COALESCE(prop_pe_trailing, 'n/a'), COALESCE(prop_pe_forward, 'n/a'),
                            COALESCE(prop_ev_ebitda, 'n/a')))),
        (l, r) -> CASE WHEN l.mc > r.mc THEN -1 WHEN l.mc < r.mc THEN 1 ELSE 0 END),
      x -> x.line), char(10))
  )
  FROM {VERTICES}
  WHERE type='Company' AND id IN (SELECT name FROM wanted)
)
""")

print(f"Created UC functions: get_company_summary, compare_companies")
print(f"Alias map covers {len(alias_pairs)} labels across {len(COMPANIES)} manifest companies + Mag7")
print()
print(spark.sql(f"SELECT `{CATALOG}`.`{SCHEMA}`.get_company_summary('Providence')").collect()[0][0][:300])


In [ ]:
# --- Smoke Tests ---
# Validate the graph tables directly via Spark SQL before relying on them as agent tools. Each test covers
# something that has actually broken here before, or that would break silently.

print("Testing graph queries...")

print("\nTest 1: a listed company still resolves by ticker")  # the legacy path
spark.sql(f"""
    SELECT id, ticker, prop_market_cap as market_cap, prop_beta as beta, prop_pe_forward as pe_forward
    FROM `{CATALOG}`.`{SCHEMA}`.graphrag_vertices
    WHERE type = 'Company' AND ticker IN ('NVDA', 'F', 'PFE')
    ORDER BY ticker
""").show(truncate=False)

# The reason this notebook now reads the manifest: these companies have no ticker and no price feed, so
# deriving the universe from ticker_data dropped them from the graph entirely.
print("\nTest 2: unlisted organisations are present as Company nodes")
spark.sql(f"""
    SELECT id, ticker, prop_market_cap as market_cap
    FROM `{CATALOG}`.`{SCHEMA}`.graphrag_vertices
    WHERE type = 'Company' AND ticker IS NULL
    ORDER BY id
""").show(truncate=False)

print("\nTest 3: ownership edges (the only edges asserting a real-world fact)")
spark.sql(f"""
    SELECT src, relationship, dst
    FROM `{CATALOG}`.`{SCHEMA}`.graphrag_edges
    WHERE relationship IN ('SUBSIDIARY_OF', 'JOINT_VENTURE_OF')
    ORDER BY src
""").show(truncate=False)

print("\nTest 4: document provenance - which companies are backed by a real filing")
spark.sql(f"""
    SELECT dst as document_source, COUNT(*) as companies
    FROM `{CATALOG}`.`{SCHEMA}`.graphrag_edges
    WHERE relationship = 'HAS_DOCUMENT_SOURCE'
    GROUP BY dst ORDER BY dst
""").show(truncate=False)

# Beta and valuation tiers must only cover companies that have the underlying number. If an unlisted
# company appears here, the null-handling in cell 5 has regressed and everything without market data is
# being labelled high-beta and premium-valued.
print("\nTest 5: risk tiers exclude companies with no fundamentals")
mislabelled = spark.sql(f"""
    SELECT e.src FROM `{CATALOG}`.`{SCHEMA}`.graphrag_edges e
    JOIN `{CATALOG}`.`{SCHEMA}`.graphrag_vertices v ON v.id = e.src AND v.type = 'Company'
    WHERE e.relationship IN ('HAS_BETA', 'HAS_VALUATION') AND v.prop_beta IS NULL
""").count()
print(f"  companies tiered without fundamentals: {mislabelled} (expected 0)")

print("\nTest 6: the UC functions resolve names as well as tickers")
for probe in ("F", "Ford Motor Company", "Providence Health", "Honda Financial"):
    out = spark.sql(
        f"SELECT `{CATALOG}`.`{SCHEMA}`.get_company_summary('{probe}')"
    ).collect()[0][0]
    print(f"  {probe:22} -> {out.splitlines()[0] if out else 'NO MATCH'}")

print("\nTest 7: corpus coverage")
spark.sql(f"""
    SELECT type, COUNT(*) as n FROM `{CATALOG}`.`{SCHEMA}`.graphrag_vertices
    GROUP BY type ORDER BY n DESC
""").show(truncate=False)

assert mislabelled == 0, "risk tiers leaked onto companies with no fundamentals - see cell 5"
print("✅ All tests passed")


In [0]:
%pip install networkx --quiet
dbutils.library.restartPython()

In [0]:
# --- Network Graph Visualization ---
# Renders the property graph as a network diagram using networkx + matplotlib.
# Shows all 7 Company hub nodes plus the N most recent TradingDay nodes per company,
# with TRADED_ON (solid) and NEXT_DAY (dashed) edges.

import sys, math
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Re-load config after Python restart
sys.path.append('..')
exec(open('../config.py').read())
CATALOG = catalog
SCHEMA = schema

# ── Parameters ──
N_DAYS = 5  # recent trading days per company to display

# ── Load graph tables ──
vertices_df = spark.table(f"{CATALOG}.{SCHEMA}.graphrag_vertices")
edges_df = spark.table(f"{CATALOG}.{SCHEMA}.graphrag_edges")

# ── Select visible nodes ──
company_pd = vertices_df.filter("type = 'Company'").select("id", "ticker").toPandas()

w_rank = Window.partitionBy("ticker").orderBy(F.col("prop_date").desc())
recent_pd = (
    vertices_df
    .filter("type = 'TradingDay'")
    .withColumn("rn", F.row_number().over(w_rank))
    .filter(f"rn <= {N_DAYS}")
    .select("id", "ticker", "prop_date", "prop_daily_return")
    .toPandas()
)

visible_ids = set(company_pd["id"].tolist() + recent_pd["id"].tolist())

# ── Filter edges to visible subgraph ──
edges_pd = edges_df.toPandas()
vis_edges = edges_pd[edges_pd["src"].isin(visible_ids) & edges_pd["dst"].isin(visible_ids)]

# ── Build networkx DiGraph ──
G = nx.DiGraph()

for _, r in company_pd.iterrows():
    G.add_node(r["id"], node_type="Company", ticker=r["ticker"], label=r["ticker"])

for _, r in recent_pd.iterrows():
    date_short = str(r["prop_date"])[:10] if r["prop_date"] else ""
    G.add_node(r["id"], node_type="TradingDay", ticker=r["ticker"], label=date_short)

for _, r in vis_edges.iterrows():
    G.add_edge(r["src"], r["dst"], relationship=r["relationship"])

# ── Colour palette ──
# Generated from a colormap rather than hand-listed, because the universe is no longer
# seven known tickers - it is whatever conf/companies.yaml contains, including
# organisations with no ticker at all (which fall back to grey).
import matplotlib.cm as cm
_symbols = sorted({t for t in company_pd["ticker"].dropna().tolist()})
palette = {
    sym: cm.tab20(i % 20)
    for i, sym in enumerate(_symbols)
}

# ── Layout: place Company nodes in a circle, spring-push TradingDays outward ──
company_ids = [n for n, d in G.nodes(data=True) if d["node_type"] == "Company"]
td_ids      = [n for n, d in G.nodes(data=True) if d["node_type"] == "TradingDay"]

ticker_to_cid = {G.nodes[n]["ticker"]: n for n in company_ids}

fixed_pos = {}
for i, cid in enumerate(sorted(company_ids)):
    angle = 2 * math.pi * i / len(company_ids)
    fixed_pos[cid] = (3 * math.cos(angle), 3 * math.sin(angle))

initial_pos = {**fixed_pos}
for n in td_ids:
    ticker = G.nodes[n]["ticker"]
    parent_cid = ticker_to_cid.get(ticker)
    if parent_cid and parent_cid in fixed_pos:
        cx, cy = fixed_pos[parent_cid]
    else:
        cx, cy = 0, 0
    initial_pos[n] = (cx + 0.3 * (hash(n) % 10 - 5), cy + 0.3 * (hash(n) % 7 - 3))

pos = nx.spring_layout(G, pos=initial_pos, fixed=company_ids, k=1.8, iterations=80, seed=42)

# ── Draw ──
fig, ax = plt.subplots(figsize=(18, 14))

traded_on = [(u, v) for u, v, d in G.edges(data=True) if d["relationship"] == "TRADED_ON"]
next_day  = [(u, v) for u, v, d in G.edges(data=True) if d["relationship"] == "NEXT_DAY"]

nx.draw_networkx_edges(G, pos, edgelist=traded_on, alpha=0.25, edge_color="#888888",
                       style="solid", arrows=True, arrowsize=12, ax=ax,
                       connectionstyle="arc3,rad=0.05")
nx.draw_networkx_edges(G, pos, edgelist=next_day, alpha=0.6, edge_color="#E74C3C",
                       style="dashed", arrows=True, arrowsize=12, width=1.5, ax=ax)

nx.draw_networkx_nodes(G, pos, nodelist=company_ids, node_size=1400,
                       node_color=[palette.get(G.nodes[n]["ticker"], "#999") for n in company_ids],
                       edgecolors="black", linewidths=2.5, ax=ax)

nx.draw_networkx_nodes(G, pos, nodelist=td_ids, node_size=350,
                       node_color=[palette.get(G.nodes[n]["ticker"], "#999") for n in td_ids],
                       alpha=0.75, edgecolors="gray", linewidths=0.5, ax=ax)

nx.draw_networkx_labels(G, pos,
                        labels={n: G.nodes[n]["label"] for n in company_ids},
                        font_size=13, font_weight="bold", ax=ax)
nx.draw_networkx_labels(G, pos,
                        labels={n: G.nodes[n]["label"] for n in td_ids},
                        font_size=7, font_color="#333333", ax=ax)

# Legend
# One patch per symbol becomes unreadable past a handful of companies, so the legend
# now explains the edge types and node kinds instead of enumerating every issuer.
handles = [
    mpatches.Patch(color="#999999", label="Unlisted organisation (no price data)"),
]
handles += [
    plt.Line2D([0], [0], color="#888888", lw=1.5, label="TRADED_ON"),
    plt.Line2D([0], [0], color="#E74C3C", lw=1.5, ls="--", label="NEXT_DAY (temporal chain)"),
]
ax.legend(handles=handles, loc="upper left", fontsize=9, framealpha=0.9, title="Legend")

total_v = vertices_df.count()
total_e = edges_df.count()

_unlisted = int(company_pd["ticker"].isna().sum())
ax.set_title(
    f"ERA Company Knowledge Graph \u2014 {len(company_ids)} Organisations "
    f"({_unlisted} unlisted) \u00d7 {N_DAYS} Recent Trading Days",
    fontsize=16, fontweight="bold", pad=20
)
ax.text(0.5, -0.02,
        f"Full graph: {total_v:,} vertices \u00b7 {total_e:,} edges  |  "
        f"Showing {G.number_of_nodes()} nodes \u00b7 {G.number_of_edges()} edges",
        transform=ax.transAxes, ha="center", fontsize=10, color="gray")

ax.axis("off")
plt.tight_layout()
plt.show()

## Setup Complete

### Artifacts Created

| Layer | Resource | Details |
|-------|----------|---------|
| **Delta Tables** | `{catalog}.{schema}.graphrag_vertices` | Company nodes (one per manifest organisation plus the Mag7) + TradingDay nodes + hub nodes for indices, sectors, tiers and document sources |
| | `{catalog}.{schema}.graphrag_edges` | TRADED_ON, NEXT_DAY, MEMBER_OF, BELONGS_TO, ownership, HAS_DOCUMENT_SOURCE, tier and correlation edges |
| **UC Functions** | `get_company_summary`, `compare_companies` | Registered as Supervisor tools by notebook 04; granted to the app service principal by notebook 06 |

### How It Connects to the Supervisor

Notebook **`04_instructor_setup_sa`** discovers UC functions by full name and registers them
as Supervisor tools. There is no MCP connection and no Databricks App in this path.

### Example Queries via the Supervisor Agent

| Query | Routed To | What Happens |
|-------|-----------|---------------|
| "Give me Ford's fundamentals and recent prices" | `get_company_summary("Ford Motor Company")` | Company node props + last 10 TradingDay nodes |
| "Compare Pfizer, Merck and Amgen" | `compare_companies("PFE,MRK,AMGN")` | Side-by-side fundamentals sorted by market cap |
| "Who owns FordDirect?" | `get_company_summary("FordDirect")` | Resolves the unlisted node; the ownership edge names Ford Motor Company |
| "What are Providence's fundamentals?" | `get_company_summary("Providence")` | Correctly reports no market data — Providence is a nonprofit with no listed equity |

### A Note on Coverage

Three counts in this graph are deliberately different, and conflating them is the mistake
this notebook is built to avoid:

* **Organisations** — everything in `conf/companies.yaml`
* **Priced companies** — those with a ticker; fewer, because nonprofits, private companies
  and debt-only registrants have none
* **Companies backed by a real filing** — fewer still on one axis and more on another:
  American Honda Finance files without being listed, while Henkel and Munich Re are listed
  abroad without filing with the SEC

Smoke tests 2, 4 and 5 assert each of these separately.

### Cleanup

```sql
DROP TABLE IF EXISTS {catalog}.{schema}.graphrag_vertices;
DROP TABLE IF EXISTS {catalog}.{schema}.graphrag_edges;
DROP FUNCTION IF EXISTS {catalog}.{schema}.get_company_summary;
DROP FUNCTION IF EXISTS {catalog}.{schema}.compare_companies;
```

### Related Notebooks

* **`00_tables_setup`** — creates the source `ticker_data` table (prerequisite)
* **`01_instructor_setup_ka`** — indexes the document corpus, including the generated documents
* **`04_instructor_setup_sa`** — creates the Supervisor and wires in these UC functions
* **`06_deploy_chatbot_app`** — grants the app service principal EXECUTE on them